In [1]:
# Kết nối đến Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# 1. Import thư viện

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
import shutil
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.mps.is_available()}")

root = "/Users/quangmanh/Project/Olympic-AI-Practice-1"

import warnings
warnings.filterwarnings("ignore")

PyTorch version: 2.8.0
MPS available: True


In [3]:
# 2. Thiết lập Seed để tái lập kết quả

def set_seed(seed=42):
    """
    Thiết lập seed cho tất cả các thư viện để đảm bảo kết quả có thể tái lập

    Args:
        seed (int): Giá trị seed cần thiết lập
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # Cho multi-GPU

    # Đảm bảo các thuật toán deterministic
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

    # Thiết lập seed cho DataLoader workers
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"Đã thiết lập seed = {seed}")

SEED = 42
set_seed(SEED)

Đã thiết lập seed = 42


In [4]:
# 3. Định nghĩa Transform cho ảnh

# Transform cho Val/Test: Chỉ resize và chuẩn hóa
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Transform cho Train: Light Augmentation (Chỉ lật ngang & lật dọc tự nhiên)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Đọc annotations từ tập train
train_csv_path = os.path.join(root, "data/train/annotations.csv")
df = pd.read_csv(train_csv_path)

# Gán nhãn tạm (0 cho không bão, 1-4 cho bão) để Stratified Split cân bằng các class
labels = df.apply(lambda row: 0 if row["is_negative"] else int(row["category_id"]), axis=1)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=labels)

print(f"Đã định nghĩa transform và chia train/val (Train: {len(train_df)}, Val: {len(val_df)})")

Đã định nghĩa transform và chia train/val (Train: 6708, Val: 1677)


In [5]:
# 4. Xây dựng Dataset Class

class StormDataset(Dataset):
    """
    Dataset cho ảnh bão

    Cấu trúc thư mục:
        root_dir/
            ├── images/
            │   ├── image1.jpg
            │   └── ...
            └── annotations.csv (chỉ có trong folder train)

    Labels:
        - 0: Không bão (is_negative=True)
        - 1: TD (Tropical Depression)
        - 2: TS (Tropical Storm)
        - 3: STS (Severe Tropical Storm)
        - 4: TY (Typhoon)
    """
    def __init__(self, root_dir, df=None, transform=None, is_test=False):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.transform = transform
        self.is_test = is_test

        if is_test:
            if df is not None:
                self.annotations = df.reset_index(drop=True)
            else:
                self.image_files = sorted([f for f in os.listdir(self.img_dir)
                                          if f.endswith((".jpg", ".jpeg", ".png"))])
                self.annotations = pd.DataFrame({"file_name": self.image_files})
        else:
            if df is not None:
                self.annotations = df.reset_index(drop=True)
            else:
                self.annotations = pd.read_csv(os.path.join(root_dir, "annotations.csv"))

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row["file_name"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, -1  # Label giả cho test set

        if row["is_negative"]:
            label = 0
        else:
            label = int(row["category_id"])

        return image, label

print("Đã định nghĩa StormDataset (hỗ trợ truyền df)")

Đã định nghĩa StormDataset (hỗ trợ truyền df)


In [6]:
# 5. Xây dựng Model CustomResNet (Residual Skip-Connections) & FocalLoss

class ResidualBlock(nn.Module):
    """
    Khối Residual cơ bản: out = ReLU(Conv(Conv(x)) + shortcut(x))
    Skip-Connection giúp gradient truyền thẳng qua mạng mà không bị suy hao.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut (Skip-connection)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + residual)
        return out

class CustomResNet(nn.Module):
    """
    Kiến trúc ResNet tùy chỉnh 4 tầng Residual: 32 -> 64 -> 128 -> 256 -> 512
    Khắc phục triệt để hiện tượng suy biến gradient của Plain CNN
    """
    def __init__(self, num_classes=5):
        super().__init__()
        # Stem ban đầu
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 224 -> 112
        )

        # 4 Stage Residual Blocks
        self.stage1 = nn.Sequential(
            ResidualBlock(32, 64, stride=2),   # 112 -> 56
            ResidualBlock(64, 64, stride=1)
        )
        self.stage2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),  # 56 -> 28
            ResidualBlock(128, 128, stride=1)
        )
        self.stage3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2), # 28 -> 14
            ResidualBlock(256, 256, stride=1)
        )
        self.stage4 = nn.Sequential(
            ResidualBlock(256, 512, stride=2), # 14 -> 7
            ResidualBlock(512, 512, stride=1)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

class FocalLoss(nn.Module):
    """
    Focal Loss: FL(pt) = - (1 - pt)^gamma * log(pt)
    Giảm trọng số loss của các mẫu dễ (Class 0: 99.8% F1)
    Dồn gradient tập trung học các class khó/ít mẫu (Class 2, Class 3)
    """
    def __init__(self, gamma=1.5, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets,
            reduction="none",
            label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Test model & loss
model = CustomResNet(num_classes=5)
criterion = FocalLoss(gamma=1.5, label_smoothing=0.05)
print(f"Đã xây dựng CustomResNet với Residual Skip-Connections!")
print(f"Tổng số parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Đã khởi tạo FocalLoss (gamma=1.5, label_smoothing=0.05)")

Đã xây dựng CustomResNet với Residual Skip-Connections!
Tổng số parameters: 11,154,213
Đã khởi tạo FocalLoss (gamma=1.5, label_smoothing=0.05)


In [7]:
# 6. Chuẩn bị DataLoader

def worker_init_fn(worker_id):
    """
    Hàm khởi tạo seed cho mỗi worker trong DataLoader
    Đảm bảo tính deterministic khi load data
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

generator = torch.Generator().manual_seed(SEED)

# Load dataset riêng cho Train (train_transform) và Val (val_transform)
train_dir = os.path.join(root, "data/train")
train_dataset = StormDataset(train_dir, df=train_df, transform=train_transform)
val_dataset = StormDataset(train_dir, df=val_df, transform=val_transform)

total_size = len(train_dataset) + len(val_dataset)
print(f"Tổng số ảnh: {total_size}")
print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}")

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    worker_init_fn=worker_init_fn,
    num_workers=0,
    pin_memory=True
)

print(f"Đã tạo DataLoader (batch_size={BATCH_SIZE})")

Tổng số ảnh: 8385
Train: 6708, Validation: 1677
Đã tạo DataLoader (batch_size=64)


In [8]:
# 7. Huấn luyện Model

# Thiết lập device
device = torch.device("mps") if torch.mps.is_available() else torch.device("cpu")
print(f"🔧 Sử dụng device: {device}")

# Khởi tạo CustomResNet, FocalLoss, optimizer
model = CustomResNet(num_classes=5).to(device)
criterion = FocalLoss(gamma=1.5, label_smoothing=0.05).to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
schedular = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

NUM_EPOCHS = 40
best_model_path = os.path.join(root, "outputs/model", "cnn_best_model.pth")
best_val_loss = float("inf")

print(f"Bắt đầu huấn luyện CustomResNet ({NUM_EPOCHS} epochs)...")

# Lưu history
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}

# Training loop
for epoch in range(NUM_EPOCHS):
    # ===== TRAINING PHASE =====
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # ===== VALIDATION PHASE =====
    model.eval()
    val_correct, val_total = 0, 0
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = 100 * val_correct / val_total

    # Lưu history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # Lưu Best Model theo Val Loss
    best_msg = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        best_msg = f" ⭐ Đã lưu Best Model (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%)"

    schedular.step()
    cur_lr = schedular.get_last_lr()[0]

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"  Learning Rate: {cur_lr:.6f}{best_msg}")

print(f"Hoàn thành huấn luyện! Best Val Loss: {best_val_loss:.4f}")
print(f"Best model đã lưu tại: {best_model_path}")

🔧 Sử dụng device: mps
Bắt đầu huấn luyện CustomResNet (40 epochs)...


Epoch 1/40 [Val]: 100%|██████████| 27/27 [00:10<00:00,  2.70it/s]


Epoch [1/40]:
  Train Loss: 0.7070, Train Acc: 56.05%
  Val Loss: 4.5102, Val Acc: 32.26%
  Learning Rate: 0.000300 ⭐ Đã lưu Best Model (Val Loss: 4.5102, Val Acc: 32.26%)


Epoch 2/40 [Val]: 100%|██████████| 27/27 [00:09<00:00,  2.76it/s]


Epoch [2/40]:
  Train Loss: 0.5846, Train Acc: 63.07%
  Val Loss: 0.6936, Val Acc: 57.48%
  Learning Rate: 0.000298 ⭐ Đã lưu Best Model (Val Loss: 0.6936, Val Acc: 57.48%)


...

Epoch [38/40]:
  Train Loss: 0.0334, Train Acc: 99.79%
  Val Loss: 0.4561, Val Acc: 79.79%
  Learning Rate: 0.000002


Epoch 39/40 [Val]: 100%|██████████| 27/27 [00:09<00:00,  2.76it/s]


Epoch [39/40]:
  Train Loss: 0.0333, Train Acc: 99.78%
  Val Loss: 0.4607, Val Acc: 79.55%
  Learning Rate: 0.000000


Epoch 40/40 [Val]: 100%|██████████| 27/27 [00:09<00:00,  2.77it/s]

Epoch [40/40]:
  Train Loss: 0.0334, Train Acc: 99.81%
  Val Loss: 0.4585, Val Acc: 79.31%
  Learning Rate: 0.000000
Hoàn thành huấn luyện! Best Val Loss: 0.3692
Best model đã lưu tại: /Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/model/cnn_best_model.pth


In [9]:
# 8. Lưu Model và Lịch sử Huấn luyện

# Lưu checkpoint epoch cuối cùng (dự phòng)
last_model_path = os.path.join(root, "outputs/model", "cnn_model_last.pth")
torch.save(model.state_dict(), last_model_path)
print(f"Đã lưu model epoch cuối tại: {last_model_path}")
print(f"Best model được lưu tại: {best_model_path}")

# Lưu history ra file CSV
history_path = os.path.join(root, "outputs/log", "training_history_20_9.csv")
history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print(f"Đã lưu lịch sử huấn luyện tại: {history_path}")

Đã lưu model epoch cuối tại: /Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/model/cnn_model_last.pth
Best model được lưu tại: /Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/model/cnn_best_model.pth
Đã lưu lịch sử huấn luyện tại: /Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/log/training_history_20_9.csv


In [10]:
# 9. Inference trên Public Test Set

print("Bắt đầu inference trên PUBLIC TEST SET...")

# Nạp trọng số từ Best Model đã lưu
best_model_path = os.path.join(root, "outputs/model", "cnn_best_model.pth")
model = CustomResNet(num_classes=5).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print(f"Model đang được eval: {best_model_path}")

# Load test dataset với val_transform
public_test_dataset = StormDataset(os.path.join(root, "data/public_test"), transform=val_transform, is_test=True)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

file_names = []
preds_is_negative = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(public_test_loader, desc="Public Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = public_test_dataset.annotations.iloc[batch_start:batch_end]["file_name"].values
        file_names.extend(batch_files)

        for p in predicted.cpu().numpy():
            if p == 0:
                preds_is_negative.append(True)
                preds_category_id.append("")
            else:
                preds_is_negative.append(False)
                preds_category_id.append(int(p))

df_public = pd.DataFrame({
    "file_name": file_names,
    "is_negative": preds_is_negative,
    "category_id": preds_category_id
})

public_csv_path = os.path.join(root, "public_cv.csv")
df_public.to_csv(public_csv_path, index=False)
print(f"Đã tạo file public_cv.csv tại {public_csv_path} với {len(df_public)} dự đoán!")

print("Preview kết quả:")
print(df_public.head(10))

Bắt đầu inference trên PUBLIC TEST SET...
Model đang được eval: /Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/model/cnn_best_model.pth


Public Test: 100%|██████████| 19/19 [00:06<00:00,  2.75it/s]

Đã tạo file public_cv.csv tại /Users/quangmanh/Project/Olympic-AI-Practice-1/public_cv.csv với 1200 dự đoán!
Preview kết quả:
                        file_name  is_negative category_id
0  201912250300_1929_4784fd1b.jpg        False           4
1  201912250600_1929_da484e8c.jpg        False           4
2  201912250900_1929_6e9f3f76.jpg        False           4
3  201912251200_1929_069f1f0e.jpg        False           3
4  201912251500_1929_1ab6db5e.jpg        False           2
5  201912260000_1929_2605a4a4.jpg        False           3
6  201912260300_1929_483edd55.jpg        False           3
7  201912260600_1929_02176738.jpg        False           4
8  201912260900_1929_1a286b61.jpg        False           4
9  201912261200_1929_d0db9605.jpg        False           4


In [11]:
negative_df = df_public[df_public["is_negative"] == False]
filename_negative = list(negative_df["file_name"])
save_images_path = "/Users/quangmanh/Project/Olympic-AI-Practice-1/outputs/images"
for image in filename_negative[:10]:
    src = os.path.join("/Users/quangmanh/Project/Olympic-AI-Practice-1/data/public_test/images", image)
    shutil.copy(src, save_images_path)

In [12]:
# 10. Inference trên Private Test Set

print("Bắt đầu inference trên PRIVATE TEST SET...")

# Load test dataset
private_test_dataset = StormDataset(os.path.join(root, "data/private_test"), transform=val_transform, is_test=True)
private_test_loader = DataLoader(
    private_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inference (model đã ở trạng thái eval và load best weights từ Cell 9)
model.eval()
file_names = []
preds_is_negative = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(private_test_loader, desc="Private Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = private_test_dataset.annotations.iloc[batch_start:batch_end]["file_name"].values
        file_names.extend(batch_files)

        for p in predicted.cpu().numpy():
            if p == 0:
                preds_is_negative.append(True)
                preds_category_id.append("")
            else:
                preds_is_negative.append(False)
                preds_category_id.append(int(p))

df_private = pd.DataFrame({
    "file_name": file_names,
    "is_negative": preds_is_negative,
    "category_id": preds_category_id
})

private_csv_path = os.path.join(root, "private_cv.csv")
df_private.to_csv(private_csv_path, index=False)
print(f"Đã tạo file private_cv.csv với {len(df_private)} dự đoán!")

print("Preview kết quả:")
print(df_private.head(10))

Bắt đầu inference trên PRIVATE TEST SET...


Private Test: 100%|██████████| 38/38 [00:13<00:00,  2.75it/s]

Đã tạo file private_cv.csv với 2398 dự đoán!
Preview kết quả:
                        file_name  is_negative category_id
0  201810280300_1826_ec69b3ac.jpg        False           4
1  201810280600_1826_a0ecedef.jpg        False           4
2  201810280900_1826_e1d45e97.jpg        False           4
3  201810281200_1826_8324ce9e.jpg        False           4
4  201810281500_1826_f48a0d0f.jpg        False           4
5  201810281800_1826_3836b841.jpg        False           4
6  201810282100_1826_19c084c5.jpg        False           4
7  201810290000_1826_e58d061d.jpg        False           3
8  201810290300_1826_caf9c524.jpg        False           4
9  201810290600_1826_4647d553.jpg        False           4
